# 📊 Github Code Analyst

## ⚙️ 설정
- 필요 SDK 설치
- 필요 라이브러리 불러오기
- 환경변수 가져오기

In [ ]:
# %pip install --upgrade pip # 필요 시
PYGITHUB_VERSION = "2.8.1"

%pip install PyGithub=={PYGITHUB_VERSION}

Note: you may need to restart the kernel to use updated packages.
Found existing installation: PyGithub 2.8.1
Uninstalling PyGithub-2.8.1:
  Successfully uninstalled PyGithub-2.8.1
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Import
import os
import logging

from abc import ABC, abstractmethod
from dataclasses import dataclass

from github import Github
from github.Auth import Token

from dotenv import load_dotenv

# 초기 설정

## env
load_dotenv(override=True)

## logger
logger_level = logging.INFO
formatter = logging.Formatter('%(asctime)s - %(levelname)s - [%(name)s] %(message)s')

logger = logging.getLogger("GithubCodeAnalyst")
logger.setLevel(logger_level)

if logger.hasHandlers():
    logger.handlers.clear()
stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)


In [ ]:
# Abstract
class _GithubLoginRequest(ABC):
    """
    Github 로그인을 위한 인증 요청 DTO
    """
    pass

class _GithubAuthenticator(ABC):
    @abstractmethod
    def githubLogin(self, request: _GithubLoginRequest) -> Github:
        pass
    
# Implementation
@dataclass
class GithubTokenLoginRequest(_GithubLoginRequest):
    """
    Github 개인 토큰을 이용한 로그인 요청 DTO
    """
    token: str

class GithubTokenAuthenticator(_GithubAuthenticator):
    def githubLogin(self, request: GithubTokenLoginRequest) -> Github:
        """
        Github 개인 토큰을 통해 로그인 및 Github 객체 생성

    
        Args:
            request (GithubTokenLoginRequest): GitHub 토큰이 포함된 DTO 객체.
        Returns:
            Github: 인증이 완료되어 API 통신에 바로 사용할 수 있는 PyGithub 클라이언트 객체.
        Raises:
            Exception: 토큰이 유효하지 않거나, GitHub API 서버와 통신할 수 없는 경우 발생합니다. 
        """
        github_token = Token(request.token)
        github = Github(auth=github_token)
        try:
            user = github.get_user()
            logger.info("[GITHUB_LOGIN] Success to login | username : " + user.login)
            return github
        except Exception as e:
            logger.error(f"[GITHUB_LOGIN] Fail to authenticate user | {e}", exc_info=True)
            raise

In [ ]:
# 로그인 전략 선택 및 객체 생성

## 토큰 불러오기
token_env_name = "GITHUB_TOKEN"
token_value = os.getenv(token_env_name)

if not token_value:
    raise RuntimeError(f"[GITHUB_LOGIN] 환경 변수 '{token_env_name}'가 설정되지 않았습니다. 확인해주세요.")

## 객체 생성
login_request = GithubTokenLoginRequest(token_value)
authenticator = GithubTokenAuthenticator()

In [ ]:
# 로그인
github = authenticator.githubLogin(login_request)